In [1]:
import os
from supabase import create_client, Client
from dotenv import load_dotenv
import yfinance as yf
import pandas as pd
from supabase import create_client
from datetime import datetime, timedelta

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("SUPABASE_URL and SUPABASE_KEY must be set in .env file")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

TICKERS = ["USDEUR=X", 'IUSA.AS', 'DIE.BR', 'FUR.AS', 'VVSM.DE', 'EMIM.AS']

def sync_portfolio():
    for ticker in TICKERS:
        # 1. Get the last recorded date for this ticker from Supabase
        response = supabase.table("StockPrices") \
            .select("Date") \
            .eq("Ticker", ticker) \
            .order("Date", desc=True) \
            .limit(1) \
            .execute()

        if response.data:
            last_date = datetime.fromisoformat(response.data[0]['Date'].replace('Z', '+00:00'))
            start_fetch = last_date + timedelta(days=1)
        else:
            # If no data exists, fetch last 5 years
            start_fetch = datetime.now() - timedelta(days=5*365)

        # 2. Fetch only new data if start_fetch is in the past
        if start_fetch.date() < datetime.now().date():
            print(f"Updating {ticker} from {start_fetch.date()}...")
            df = yf.download(ticker, start=start_fetch, interval="1d")['Close'].reset_index()
            if not df.empty:
                # Prepare data for Supabase insert
                records = []
                for _, row in df.iterrows():
                    records.append({
                        "Date": row['Date'].isoformat(),
                        "Ticker": ticker,
                        "StockPrice": row[ticker]
                    })
                
                # 3. Bulk insert to Supabase
                print('test')
                supabase.table("StockPrices").insert(records).execute()

# Run the sync
sync_portfolio()

Updating IUSA.AS from 2021-01-18...


[*********************100%***********************]  1 of 1 completed


test
Updating DIE.BR from 2021-01-18...


[*********************100%***********************]  1 of 1 completed


test
Updating FUR.AS from 2021-01-18...


[*********************100%***********************]  1 of 1 completed


test


In [8]:
ticker = yf.Ticker('inga.as').get_news()
ticker

[{'id': '6c693463-8afc-3c1b-8112-abd126cd89b1',
  'content': {'id': '6c693463-8afc-3c1b-8112-abd126cd89b1',
   'contentType': 'STORY',
   'title': 'European Dividend Stocks To Watch In January 2026',
   'description': '',
   'summary': "As 2026 begins, European markets are showing signs of optimism, with major indices like the STOXX Europe 600 and Germany's DAX experiencing notable gains amid positive economic indicators and a favorable interest rate environment. In this context, dividend stocks in Europe are attracting attention for their potential to offer stable income streams and resilience against market fluctuations.",
   'pubDate': '2026-01-15T10:32:13Z',
   'displayTime': '2026-01-15T10:32:13Z',
   'isHosted': True,
   'bypassModal': False,
   'previewUrl': None,
   'thumbnail': {'originalUrl': 'https://media.zenfs.com/en/simply_wall_st__316/064d0b7911046f8efd85160776a9b33b',
    'originalWidth': 1194,
    'originalHeight': 432,
    'caption': '',
    'resolutions': [{'url': 'h